# 06 · Window functions

`groupBy` **collapses** each group into one row. A **window function** keeps **every row**, but lets each
row look at the other rows in its group ("window"): its rank, the previous row's value, a running total…

```
groupBy("dept").sum()            window partitionBy("dept")
┌──────┬─────┐                   ┌──────┬──────┬────────┬──────────┐
│ dept │ sum │                   │ dept │ name │ amount │ dept_sum │
│ Eng  │ 1300│                   │ Eng  │ Eve  │ 500    │ 1300     │
│ Sales│ 900 │                   │ Eng  │ Fay  │ 400    │ 1300     │  ← every row kept
└──────┴─────┘                   │ ...  │      │        │          │
```
A window is defined by:
* `partitionBy(...)` — which rows belong together (like groupBy)
* `orderBy(...)` — the order within each partition (needed for ranking, lag, running totals)
* optionally a **frame** — which rows around the current row to include (`rowsBetween`)

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
sales = spark.createDataFrame([
    ("Sales", "Ann", 100), ("Sales", "Ben", 300), ("Sales", "Cat", 300), ("Sales", "Dev", 200),
    ("Eng", "Eve", 500), ("Eng", "Fay", 400), ("Eng", "Gus", 400)],
    "dept string, name string, amount int")

w = Window.partitionBy("dept").orderBy(F.desc("amount"))
sales.select("*",
             F.row_number().over(w).alias("row_number"),
             F.rank().over(w).alias("rank"),
             F.dense_rank().over(w).alias("dense_rank")).show()

* `row_number` — 1, 2, 3, 4 … always unique (ties broken arbitrarily — add more `orderBy` columns to make it deterministic)
* `rank` — ties share a rank, then it **skips** (1, 2, 2, 4)
* `dense_rank` — ties share a rank, **no gaps** (1, 2, 2, 3)

## Partition-only windows: compare each row to its group

In [ ]:
by_dept = Window.partitionBy("dept")
sales.select("*",
             F.sum("amount").over(by_dept).alias("dept_total"),
             F.round(F.col("amount") / F.sum("amount").over(by_dept) * 100, 1).alias("pct_of_dept"),
             (F.col("amount") - F.avg("amount").over(by_dept)).alias("vs_dept_avg")).show()

## `lag` and `lead`: look at the previous / next row

In [ ]:
daily = spark.createDataFrame(
    [("2024-01-01", 10), ("2024-01-02", 14), ("2024-01-03", 9), ("2024-01-04", 20)], "day string, visits int")
w = Window.orderBy("day")
daily.select("*",
             F.lag("visits").over(w).alias("prev"),
             F.lead("visits").over(w).alias("next"),
             (F.col("visits") - F.lag("visits").over(w)).alias("change")).show()

A window **without `partitionBy`** puts all rows into a single partition (Spark warns you about it).
Fine for small data like a daily series; avoid on huge tables.

## Running totals and moving averages: frames
`rowsBetween(start, end)` sets which rows the aggregate sees, relative to the current row (0).
`Window.unboundedPreceding` = from the very first row.

In [ ]:
running = Window.orderBy("day").rowsBetween(Window.unboundedPreceding, Window.currentRow)
moving3 = Window.orderBy("day").rowsBetween(-1, 1)      # previous, current, next
daily.select("*",
             F.sum("visits").over(running).alias("running_total"),
             F.round(F.avg("visits").over(moving3), 2).alias("moving_avg_3")).show()

## The famous "top N per group" pattern
Number the rows within each group, then filter. You'll write this a lot (and get asked it in interviews):

In [ ]:
w = Window.partitionBy("dept").orderBy(F.desc("amount"), "name")
(sales.withColumn("rn", F.row_number().over(w))
      .filter(F.col("rn") == 1)
      .drop("rn")
      .show())

## Now on the shop data

In [ ]:
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA}/products.csv", header=True, inferSchema=True)
completed = (orders.filter(F.col("status") == "completed")
                   .join(products, "product_id")
                   .withColumn("revenue", F.col("quantity") * F.col("price")))

### Exercise 1
Create `first_orders`: each customer's **earliest** order. Columns **`customer_id`, `order_id`, `order_ts`**. Break ties on `order_ts` with the lower `order_id`. Use all orders (any status).

In [ ]:
first_orders = None  # TODO

In [ ]:
# Check — run me after your answer
assert first_orders.columns == ["customer_id", "order_id", "order_ts"]
assert first_orders.count() == 180
assert first_orders.filter(F.col("customer_id") == 1).first().order_id == 898
print("Correct!")

### Exercise 2
Create `top_product_per_cat`: using `completed`, the single product with the **highest total revenue in each category**. Columns **`category`, `product_id`, `revenue`**.

*Hint:* aggregate first (revenue per product), then rank within category.

In [ ]:
top_product_per_cat = None  # TODO

In [ ]:
# Check — run me after your answer
assert top_product_per_cat.columns == ["category", "product_id", "revenue"]
assert sorted(r.product_id for r in top_product_per_cat.collect()) == [5, 8, 15, 19, 25]
print("Correct!")

### Exercise 3
Create `daily_rev` from `completed`: one row per day with columns **`date`, `revenue`, `prev_revenue`, `change`** (`revenue - prev_revenue`), ordered by date.

In [ ]:
daily_rev = None  # TODO

In [ ]:
# Check — run me after your answer
import datetime
rows = daily_rev.collect()
assert daily_rev.columns == ["date", "revenue", "prev_revenue", "change"]
assert len(rows) == 366 and rows[0].prev_revenue is None
day = daily_rev.filter(F.col("date") == datetime.date(2024, 3, 15)).first()
assert abs(day.change - 1439.5) < 0.05, day
print("Correct!")

### Exercise 4
Create `with_running`: `daily_rev` plus **`cum_revenue`**, the running total of revenue up to and including each day.

In [ ]:
with_running = None  # TODO

In [ ]:
# Check — run me after your answer
last = with_running.orderBy(F.desc("date")).first()
assert abs(last.cum_revenue - 1401196.05) < 0.5, last.cum_revenue
print("Correct!")

---
Stuck? Worked answers (with outputs) are in [`../solutions/06_window_functions.ipynb`](../solutions/06_window_functions.ipynb).